In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
# =========================
# Konfigurasi Utama
# =========================

BASE_DIR = Path.cwd()
PRIMARY_DATA_DIR = BASE_DIR / 'data' / 'raw' / 'primer'
SECONDARY_DATA_DIR = BASE_DIR / 'data' / 'raw' / 'sekunder'
SPLIT_DIR = BASE_DIR / 'data' / 'splits'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
USE_PREPARED_DATA = True

# Nama kelas HARUS sama dengan nama folder kelas
CLASS_NAMES = ['leaf curl', 'leaf spot', 'yellowish', 'healthy leaf']

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
INITIAL_LR = 1e-4
EPOCHS = 40  # bisa 30-50

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-6

MODEL_DIR = BASE_DIR / 'models' / 'checkpoints'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
(BASE_DIR / 'models' / 'final').mkdir(parents=True, exist_ok=True)
(BASE_DIR / 'reports' / 'figures').mkdir(parents=True, exist_ok=True)
(BASE_DIR / 'reports' / 'metrics').mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print('Project base dir   :', BASE_DIR)
print('Primary dir exists :', PRIMARY_DATA_DIR.exists())
print('Secondary dir exists:', SECONDARY_DATA_DIR.exists())
print('Split dir exists   :', SPLIT_DIR.exists())
print('Processed dir exists:', PROCESSED_DIR.exists())
print('Classes:', CLASS_NAMES)
print('Epochs:', EPOCHS, '| LR:', INITIAL_LR)

In [ ]:
# =========================
# Utilitas Dataset
# =========================

VALID_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

CLASS_ALIASES = {
    'leaf curl': ['leaf curl', 'leaf_curl', 'leaf-curl', 'curl'],
    'leaf spot': ['leaf spot', 'leaf_spot', 'leaf-spot', 'spot'],
    'yellowish': ['yellowish', 'yellow', 'kuning'],
    'healthy leaf': ['healthy leaf', 'healthy_leaf', 'healthy-leaf', 'healthy', 'sehat'],
}


def normalize_text(text: str) -> str:
    text = text.strip().lower()
    text = text.replace('_', ' ').replace('-', ' ')
    return ' '.join(text.split())


def infer_label_from_path(path_obj: Path):
    parts_norm = [normalize_text(p) for p in path_obj.parts]

    for class_name in CLASS_NAMES:
        aliases = CLASS_ALIASES.get(class_name, [class_name])
        aliases_norm = [normalize_text(a) for a in aliases]
        if any(alias in parts_norm for alias in aliases_norm):
            return class_name

    return None


def collect_images_from_root(root_dir: Path, source_name: str) -> pd.DataFrame:
    records = []
    if not root_dir.exists():
        print(f'[Warning] Root dataset tidak ditemukan: {root_dir}')
        return pd.DataFrame(columns=['filepath', 'label', 'source'])

    image_files = [
        p for p in root_dir.rglob('*')
        if p.is_file() and p.suffix.lower() in VALID_EXT
    ]

    skipped = 0
    for fp in image_files:
        label = infer_label_from_path(fp)
        if label is None:
            skipped += 1
            continue

        records.append(
            {
                'filepath': str(fp),
                'label': label,
                'source': source_name,
            }
        )

    if skipped > 0:
        print(f'[Info] {source_name}: {skipped} file tidak dipakai karena label tidak terdeteksi dari path folder.')

    return pd.DataFrame(records)


def show_distribution(df: pd.DataFrame, title: str = 'Distribusi Kelas'):
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df, x='label', order=CLASS_NAMES)
    plt.title(title)
    plt.xticks(rotation=20)
    plt.show()


In [ ]:
# =========================
# Split Data Stratified: 70/15/15
# =========================

label_to_idx = {label: i for i, label in enumerate(CLASS_NAMES)}
idx_to_label = {i: label for label, i in label_to_idx.items()}

def load_manifest(csv_path):
    df = pd.read_csv(csv_path)
    if 'source' not in df.columns:
        df['source'] = csv_path.stem
    return df

processed_manifests = {
    'train': PROCESSED_DIR / 'train.csv',
    'validation': PROCESSED_DIR / 'validation.csv',
    'test': PROCESSED_DIR / 'test.csv',
}
split_manifests = {
    'train': SPLIT_DIR / 'train.csv',
    'validation': SPLIT_DIR / 'validation.csv',
    'test': SPLIT_DIR / 'test.csv',
}

def add_label_index(df):
    df = df.copy()
    df['label_idx'] = df['label'].map(label_to_idx)
    return df

use_processed = USE_PREPARED_DATA and all(path.exists() for path in processed_manifests.values())
use_split_manifests = USE_PREPARED_DATA and (not use_processed) and all(path.exists() for path in split_manifests.values())

if use_processed:
    train_df = add_label_index(load_manifest(processed_manifests['train']))
    val_df = add_label_index(load_manifest(processed_manifests['validation']))
    test_df = add_label_index(load_manifest(processed_manifests['test']))
    data_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    print('Loaded prepared data from data/processed')
elif use_split_manifests:
    train_df = add_label_index(load_manifest(split_manifests['train']))
    val_df = add_label_index(load_manifest(split_manifests['validation']))
    test_df = add_label_index(load_manifest(split_manifests['test']))
    data_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    print('Loaded prepared data from data/splits')
else:
    primary_df = collect_images_from_root(PRIMARY_DATA_DIR, source_name='primer')
    secondary_df = collect_images_from_root(SECONDARY_DATA_DIR, source_name='sekunder')
    data_df = pd.concat([primary_df, secondary_df], ignore_index=True)

    if len(data_df) == 0:
        raise ValueError('Data tidak ditemukan. Periksa path dataset dan nama folder kelas.')

    train_df, temp_df = train_test_split(
        data_df,
        test_size=(1.0 - TRAIN_RATIO),
        random_state=SEED,
        stratify=data_df['label'],
    )

    val_ratio_adjusted = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
    val_df, test_df = train_test_split(
        temp_df,
        test_size=(1.0 - val_ratio_adjusted),
        random_state=SEED,
        stratify=temp_df['label'],
    )

    train_df = add_label_index(train_df)
    val_df = add_label_index(val_df)
    test_df = add_label_index(test_df)
    print('Created split directly from raw dataset')

if len(data_df) == 0:
    raise ValueError('Data tidak ditemukan. Periksa path dataset dan folder split/processed.')

print(f'Train size: {len(train_df)} ({len(train_df)/len(data_df):.2%})')
print(f'Val size  : {len(val_df)} ({len(val_df)/len(data_df):.2%})')
print(f'Test size : {len(test_df)} ({len(test_df)/len(data_df):.2%})')

print('\nDistribusi train:')
display(train_df['label'].value_counts())
print('\nDistribusi val:')
display(val_df['label'].value_counts())
print('\nDistribusi test:')
display(test_df['label'].value_counts())

show_distribution(data_df, 'Distribusi Kelas (Gabungan)')

In [ ]:
# =========================
# Preprocessing dan Augmentasi
# =========================

augmenter = keras.Sequential(
    [
        layers.RandomRotation(factor=(-0.083, 0.083)),  # ~ -30 hingga +30 derajat
        layers.RandomFlip(mode='horizontal_and_vertical'),
        layers.RandomZoom(height_factor=0.15, width_factor=0.15),
        layers.RandomTranslation(height_factor=0.10, width_factor=0.10),
    ],
    name='augmentation',
)


def decode_and_resize(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0  # normalisasi piksel ke [0, 1]
    return img, label


def build_dataset(df, training=False):
    paths = df['filepath'].values
    labels = df['label_idx'].values

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.map(lambda x, y: (augmenter(x, training=True), y), num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


train_ds = build_dataset(train_df, training=True)
val_ds = build_dataset(val_df, training=False)
test_ds = build_dataset(test_df, training=False)

print('Dataset pipeline siap.')

In [ ]:
# =========================
# Definisi Model
# =========================

NUM_CLASSES = len(CLASS_NAMES)


def build_mobilenetv2_model(train_base=False):
    inputs = keras.Input(shape=(*IMG_SIZE, 3), name='image_input')

    x = layers.Lambda(lambda t: tf.keras.applications.mobilenet_v2.preprocess_input(t * 255.0))(inputs)
    base = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights='imagenet',
        input_shape=(*IMG_SIZE, 3),
    )
    base.trainable = train_base

    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='MobileNetV2')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


def build_efficientnetb0_model(train_base=False):
    inputs = keras.Input(shape=(*IMG_SIZE, 3), name='image_input')

    x = layers.Lambda(lambda t: t * 255.0)(inputs)
    base = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(*IMG_SIZE, 3),
    )
    base.trainable = train_base

    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='EfficientNetB0')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


def build_feature_fusion_model(train_base=False):
    inputs = keras.Input(shape=(*IMG_SIZE, 3), name='image_input')

    # Branch 1: MobileNetV2
    mobile_in = layers.Lambda(lambda t: tf.keras.applications.mobilenet_v2.preprocess_input(t * 255.0))(inputs)
    mobile_base = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights='imagenet',
        input_shape=(*IMG_SIZE, 3),
    )
    mobile_base.trainable = train_base
    mobile_feat = mobile_base(mobile_in, training=False)
    mobile_feat = layers.GlobalAveragePooling2D()(mobile_feat)

    # Branch 2: EfficientNetB0
    eff_in = layers.Lambda(lambda t: t * 255.0)(inputs)
    eff_base = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(*IMG_SIZE, 3),
    )
    eff_base.trainable = train_base
    eff_feat = eff_base(eff_in, training=False)
    eff_feat = layers.GlobalAveragePooling2D()(eff_feat)

    # Feature Fusion
    fused = layers.Concatenate(name='feature_fusion')([mobile_feat, eff_feat])
    fused = layers.BatchNormalization()(fused)
    fused = layers.Dense(512, activation='relu')(fused)
    fused = layers.Dropout(0.4)(fused)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(fused)

    model = keras.Model(inputs, outputs, name='MobileNetV2_EfficientNetB0_Fusion')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


MODEL_BUILDERS = {
    'MobileNetV2': build_mobilenetv2_model,
    'EfficientNetB0': build_efficientnetb0_model,
    'Fusion': build_feature_fusion_model,
}

print('Model builders siap:', list(MODEL_BUILDERS.keys()))

In [ ]:
# Preview arsitektur model fusion
fusion_model = build_feature_fusion_model(train_base=False)
fusion_model.summary()

In [ ]:
# =========================
# Fungsi Training dan Visualisasi
# =========================

def get_callbacks(model_name: str, stage: str = 'stage1'):
    ckpt_path = MODEL_DIR / f'{model_name}_{stage}_best.keras'
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(ckpt_path),
            monitor='val_accuracy',
            mode='max',
            save_best_only=True,
            verbose=1,
        ),
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=7,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1,
        ),
    ]
    return callbacks, ckpt_path


def merge_histories(history_stage1, history_stage2=None):
    if history_stage2 is None:
        return history_stage1

    merged = {}
    for key in history_stage1.history.keys():
        merged[key] = history_stage1.history.get(key, []) + history_stage2.history.get(key, [])

    class SimpleHistory:
        pass

    h = SimpleHistory()
    h.history = merged
    return h


def _set_trainable_ratio(backbone: keras.Model, unfreeze_ratio: float = 0.2):
    n_layers = len(backbone.layers)
    n_unfreeze = max(1, int(n_layers * unfreeze_ratio))
    cutoff = n_layers - n_unfreeze

    for i, layer in enumerate(backbone.layers):
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = i >= cutoff


def enable_stage2_finetune(model, model_name: str, fine_tune_lr: float, unfreeze_ratio: float = 0.2):
    target_keywords = {
        'MobileNetV2': ['mobilenetv2'],
        'EfficientNetB0': ['efficientnetb0'],
        'Fusion': ['mobilenetv2', 'efficientnetb0'],
    }

    keywords = target_keywords.get(model_name, [])
    tuned_count = 0

    for layer in model.layers:
        if isinstance(layer, keras.Model):
            lname = layer.name.lower()
            if any(k in lname for k in keywords):
                layer.trainable = True
                _set_trainable_ratio(layer, unfreeze_ratio=unfreeze_ratio)
                tuned_count += 1

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=fine_tune_lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )

    print(f'[Stage-2] Backbone yang di-fine-tune untuk {model_name}: {tuned_count}')
    return model


def plot_history(history, model_name: str):
    hist = history.history

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(hist['accuracy'], label='train_acc')
    plt.plot(hist['val_accuracy'], label='val_acc')
    plt.title(f'{model_name} - Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(hist['loss'], label='train_loss')
    plt.plot(hist['val_loss'], label='val_loss')
    plt.title(f'{model_name} - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# =========================
# Fungsi Evaluasi
# =========================

def evaluate_model(model, test_ds, test_df, model_name: str):
    y_true = test_df['label_idx'].values

    y_prob = model.predict(test_ds, verbose=1)
    y_pred = np.argmax(y_prob, axis=1)

    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )

    print(f'\n===== {model_name} =====')
    print(f'Accuracy : {acc:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall   : {recall:.4f}')
    print(f'F1-score : {f1:.4f}')

    print('\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
    )
    plt.title(f'Confusion Matrix - {model_name}')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.xticks(rotation=20)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    return {
        'Model': model_name,
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1,
    }

In [ ]:
# =========================
# Training Semua Model (2 Tahap)
# =========================

TRAIN_TARGETS = ['MobileNetV2', 'EfficientNetB0', 'Fusion']
DO_STAGE2_FINETUNE = True

EPOCHS_STAGE1 = max(1, int(EPOCHS * 0.6))
EPOCHS_STAGE2 = max(1, EPOCHS - EPOCHS_STAGE1)
FINE_TUNE_LR = INITIAL_LR * 0.1
UNFREEZE_RATIO = 0.20  # buka 20% layer paling akhir backbone

print('Konfigurasi training:')
print(f'- Stage-1 epoch: {EPOCHS_STAGE1}')
print(f'- Stage-2 epoch: {EPOCHS_STAGE2}')
print(f'- Stage-2 LR   : {FINE_TUNE_LR}')
print(f'- Unfreeze ratio: {UNFREEZE_RATIO}')

histories = {}
trained_models = {}
results = []

for model_name in TRAIN_TARGETS:
    print('\n' + '=' * 70)
    print(f'Training {model_name}')
    print('=' * 70)

    # Stage-1: train head classifier dengan backbone freeze
    model = MODEL_BUILDERS[model_name](train_base=False)
    cb1, ckpt1 = get_callbacks(model_name, stage='stage1')

    print(f'[Stage-1] {model_name} mulai training...')
    hist1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS_STAGE1,
        callbacks=cb1,
        verbose=1,
    )

    if ckpt1.exists():
        model = keras.models.load_model(ckpt1)

    hist_merged = hist1

    # Stage-2: fine-tuning sebagian backbone
    if DO_STAGE2_FINETUNE:
        model = enable_stage2_finetune(
            model,
            model_name=model_name,
            fine_tune_lr=FINE_TUNE_LR,
            unfreeze_ratio=UNFREEZE_RATIO,
        )

        cb2, ckpt2 = get_callbacks(model_name, stage='stage2')

        print(f'[Stage-2] {model_name} fine-tuning mulai...')
        hist2 = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS_STAGE2,
            callbacks=cb2,
            verbose=1,
        )

        if ckpt2.exists():
            model = keras.models.load_model(ckpt2)

        hist_merged = merge_histories(hist1, hist2)

    histories[model_name] = hist_merged
    trained_models[model_name] = model

    plot_history(hist_merged, model_name)

    metrics = evaluate_model(model, test_ds, test_df, model_name)
    results.append(metrics)

results_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
print('\nPerbandingan performa model:')
display(results_df)

In [ ]:
# =========================
# Simpan Ringkasan Hasil
# =========================

if len(results) > 0:
    results_path = Path.cwd() / 'reports' / 'metrics' / 'hasil_perbandingan_model.csv'
    results_df.to_csv(results_path, index=False)
    print('Hasil evaluasi disimpan ke:', results_path.resolve())

    best_model_name = results_df.iloc[0]['Model']
    best_model_path = Path.cwd() / 'models' / 'final' / 'best_model.keras'
    trained_models[best_model_name].save(best_model_path)
    print('Model terbaik saat ini:', best_model_name)
    print('Model terbaik disimpan ke:', best_model_path.resolve())
else:
    print('Belum ada hasil. Jalankan sel training terlebih dahulu.')